In [1]:
import os
import keras
from keras import layers, models, datasets
import pickle

2026-05-08 17:58:28.911732: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-08 17:58:28.913319: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-08 17:58:29.035034: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-08 17:58:30.454952: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off,

In [2]:
os.getcwd()

'/home/bluefox/NN/Project1/notebooks/experiment'

In [3]:
os.chdir('/home/bluefox/NN/Project1')

In [4]:

(x_train, y_train), (x_test, y_test) = datasets.imdb.load_data(num_words=20000, path='imdb.npz', seed=1337, cache_dir='data/processed')


/home/bluefox/NN/Project1/.venv/lib/python3.13/site-packages/numpy/lib/_format_impl.py:838: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  array = pickle.load(fp, **pickle_kwargs)


In [5]:
x_train = keras.preprocessing.sequence.pad_sequences(x_train, maxlen=500)
x_test = keras.preprocessing.sequence.pad_sequences(x_test, maxlen=500)

In [6]:
with open('data/processed/imdb_train.pkl', 'wb') as f:
    pickle.dump((x_train, y_train), f)
with open('data/processed/imdb_test.pkl', 'wb') as f:
    pickle.dump((x_test, y_test), f)

In [7]:
x_train.shape, y_train.shape, x_test.shape, y_test.shape

((25000, 500), (25000,), (25000, 500), (25000,))

In [8]:
x_train[0].shape, y_train[0], x_test[0].shape, y_test[0]

((500,), np.int64(1), (500,), np.int64(0))

In [9]:
text_input = keras.Input(shape=(None,), dtype='int32', name='text')
embedding_layer = layers.Embedding(input_dim=20000, output_dim=128, name='embedding')(text_input)
dropout_layer = layers.Dropout(0.5, name='dropout')(embedding_layer)

2026-05-08 17:59:18.513442: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [10]:
conv_layer = layers.Conv1D(128, 7, activation='relu', name='conv')(dropout_layer)
conv_layer2 = layers.Conv1D(128, 7, activation='relu', name='conv2')(conv_layer)
pool_layer = layers.GlobalMaxPooling1D(name='global_max_pooling')(conv_layer2)

In [11]:
dense_layer = layers.Dense(128, activation='relu', name='dense')(pool_layer)
dropout_layer2 = layers.Dropout(0.5, name='dropout2')(dense_layer)
output_layer = layers.Dense(1, activation='sigmoid', name='output')(dropout_layer2)

In [12]:
model = models.Model(inputs=text_input, outputs=output_layer)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [15]:
model.save('models/architecture/untrained_demo_model.keras')